# Fine-tune Gemma 2 2B with WildChat and OpenAssistant (Colab)

This notebook fine-tunes Google's **Gemma 2 2B IT** (`google/gemma-2-2b-it`) with 4-bit loading and LoRA. It uses a small, configurable sample by default so it can run on a Colab GPU; it does not train a model from scratch. The result is a LoRA adapter you can download or copy to Google Drive.

**Before running:** select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). First accept Google's Gemma terms and request/enable access on [the model page](https://huggingface.co/google/gemma-2-2b-it) while signed in to Hugging Face. Review the datasets' licenses, terms, and privacy considerations: [WildChat-1M](https://huggingface.co/datasets/allenai/WildChat-1M) and [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1). The notebook filters to English and applies basic email/phone redaction, but automated redaction is imperfect. Do not use the resulting model to reproduce personal or sensitive information. Increase the sample size only after checking GPU capacity and dataset terms. Public repositories need no login; if a dataset or model instead returns an authentication/permission error, the notebook starts Hugging Face login and retries. Login does not itself grant Gemma access; the account must have accepted the model terms.

In [ ]:
%pip -q install "transformers>=5.0,<6" "huggingface_hub>=1.3,<2" "datasets>=3.2,<4" "peft>=0.18,<1" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1"

In [ ]:
import os
import re
import random

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          Trainer, TrainingArguments)

def with_hf_auth_retry(operation, resource):
    try:
        return operation()
    except HfHubHTTPError as error:
        status = getattr(getattr(error, "response", None), "status_code", None)
        if status not in {401, 403} and not isinstance(error, GatedRepoError):
            raise
        print(f"Hugging Face access to {resource} requires authentication.")
        print("A browser/device code will be shown. Sign in at Hugging Face and enter that code.")
        login()
        return operation()

if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU in Colab: Runtime → Change runtime type → GPU.")

MODEL_ID = "google/gemma-2-2b-it"
MAX_PER_DATASET = 1000  # Increase gradually if your GPU/runtime has room.
MAX_SEQ_LENGTH = 768
SEED = 42
OUTPUT_DIR = "/content/superbot-lora"
random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

## Load and normalize conversations

WildChat is streamed and capped; OpenAssistant is loaded from its train split and reconstructed from message-parent links. Only English conversations with user and assistant turns are kept. OpenAssistant is filtered to non-deleted, accepted, top-ranked assistant messages where the dataset provides those labels.

In [ ]:
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?<!\w)(?:\+?\d[\d().\-\s]{7,}\d)(?!\w)")

def clean_text(value):
    if not isinstance(value, str):
        return ""
    value = EMAIL_RE.sub("[EMAIL REDACTED]", value)
    return PHONE_RE.sub("[PHONE REDACTED]", value).strip()

def normalize_turns(turns, role_map=None, max_messages=12):
    role_map = role_map or {}
    messages = []
    for turn in turns:
        role = role_map.get(turn.get("role"), turn.get("role"))
        content = clean_text(turn.get("content", turn.get("text", "")))
        if role in {"user", "assistant"} and content:
            messages.append({"role": role, "content": content})
    messages = messages[-max_messages:]
    while messages and messages[0]["role"] != "user":
        messages.pop(0)
    if len(messages) >= 2 and messages[-1]["role"] == "assistant":
        return messages
    return None

def collect_wildchat_rows():
    rows = []
    wildchat = load_dataset("allenai/WildChat-1M", split="train", streaming=True)
    seen = set()
    for row in wildchat:
        if row.get("language") != "English" or row.get("toxic") is True:
            continue
        turns = row.get("conversation")
        if not isinstance(turns, list):
            continue
        messages = normalize_turns(turns)
        if not messages:
            continue
        fingerprint = tuple((m["role"], m["content"]) for m in messages)
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        rows.append({"messages": messages})
        if len(rows) >= MAX_PER_DATASET:
            break
    return rows

wildchat_rows = with_hf_auth_retry(
    collect_wildchat_rows, "allenai/WildChat-1M"
)

def collect_oasst_rows():
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    oasst_by_id = {row["message_id"]: row for row in oasst}
    rows = []
    seen = set()
    for row in oasst:
        if row.get("role") != "assistant" or row.get("lang") != "en":
            continue
        if row.get("deleted") is True or row.get("review_result") is False:
            continue
        if row.get("rank") not in (None, 0):
            continue
        chain = []
        current = row
        while current is not None and len(chain) < MAX_SEQ_LENGTH:
            chain.append(current)
            parent_id = current.get("parent_id")
            current = oasst_by_id.get(parent_id) if parent_id else None
        chain.reverse()
        turns = [
            {"role": "user" if item.get("role") == "prompter" else item.get("role"),
             "content": item.get("text", "")}
            for item in chain
        ]
        messages = normalize_turns(turns)
        if not messages:
            continue
        fingerprint = tuple((m["role"], m["content"]) for m in messages)
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        rows.append({"messages": messages})
        if len(rows) >= MAX_PER_DATASET:
            break
    return rows

oasst_rows = with_hf_auth_retry(collect_oasst_rows, "OpenAssistant/oasst1")

examples = wildchat_rows + oasst_rows
random.shuffle(examples)
if len(examples) < 20:
    raise RuntimeError(f"Only {len(examples)} usable examples loaded; check dataset access/schemas.")
print(f"WildChat: {len(wildchat_rows):,}; OpenAssistant: {len(oasst_rows):,}; total: {len(examples):,}")
print("Example roles:", [m["role"] for m in examples[0]["messages"]])

## Tokenize with the model's chat template

The notebook keeps a small held-out split for monitoring. The simple causal-LM objective trains on the formatted conversation tokens; it is a basic fine-tune, not a safety guarantee or a substitute for evaluation.

In [ ]:
tokenizer = with_hf_auth_retry(
    lambda: AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True), MODEL_ID
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(example):
    # Gemma's chat template supports user/assistant dialogue, not system roles.
    # Keep training examples restricted to the supported roles.
    messages = [
        {"role": message["role"], "content": message["content"]}
        for message in example["messages"]
        if message.get("role") in {"user", "assistant"}
    ]
    while messages and messages[0]["role"] != "user":
        messages.pop(0)
    if len(messages) < 2 or messages[-1]["role"] != "assistant":
        raise ValueError("Chat example must contain a user message and an assistant reply.")
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    token_ids = encoded["input_ids"] if isinstance(encoded, dict) else encoded
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()
    if token_ids and isinstance(token_ids[0], list):
        if len(token_ids) != 1:
            raise ValueError("Expected one token sequence for each conversation.")
        token_ids = token_ids[0]
    if not token_ids or not all(isinstance(token_id, int) for token_id in token_ids):
        raise TypeError("Chat template must return a flat list of integer token IDs.")
    return {"input_ids": token_ids}

raw_dataset = Dataset.from_list(examples)
split = raw_dataset.train_test_split(test_size=0.02, seed=SEED)
train_dataset = split["train"].map(
    tokenize_example, remove_columns=["messages"], load_from_cache_file=False
)
eval_dataset = split["test"].map(
    tokenize_example, remove_columns=["messages"], load_from_cache_file=False
)
for row in (train_dataset[0], eval_dataset[0]):
    ids = row.get("input_ids")
    if not isinstance(ids, list) or not ids or not all(isinstance(item, int) for item in ids):
        raise TypeError(f"Expected input_ids to be a flat list of integers; got {type(ids).__name__}: {str(ids)[:160]}")
print(f"Train: {len(train_dataset):,}; validation: {len(eval_dataset):,}")

## Load the quantized model and attach LoRA

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = with_hf_auth_retry(
    lambda: AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization,
        device_map="auto",
    ),
    MODEL_ID,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_steps=max(1, int(len(train_dataset) / 8 * 0.03)),
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)
def collate_chat_batch(features):
    normalized = []
    for feature in features:
        token_ids = feature.get("input_ids")
        if isinstance(token_ids, dict):
            token_ids = token_ids.get("input_ids")
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()
        if isinstance(token_ids, list) and token_ids and isinstance(token_ids[0], list):
            if len(token_ids) != 1:
                raise ValueError("Each training feature must contain exactly one token sequence.")
            token_ids = token_ids[0]
        if not isinstance(token_ids, list) or not token_ids or not all(isinstance(item, int) for item in token_ids):
            raise TypeError(f"Malformed input_ids reached batch collation: {str(token_ids)[:200]}")
        normalized.append({"input_ids": token_ids})
    batch = tokenizer.pad(normalized, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_chat_batch,
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)

## Try the fine-tuned model

In [ ]:
model.eval()
test_messages = [{"role": "user", "content": "Hi! Can you help me make a simple plan for today?"}]
prompt_ids = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)
with torch.no_grad():
    generated = model.generate(
        prompt_ids,
        max_new_tokens=160,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(generated[0][prompt_ids.shape[1]:], skip_special_tokens=True))

## Download the adapter

This downloads a ZIP containing the LoRA adapter and tokenizer. The adapter is not a standalone model; load it together with the same base model. You can also uncomment the Drive cells to preserve it after the Colab runtime ends.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive("superbot-lora", "zip", OUTPUT_DIR)
files.download(archive_path)

In [ ]:
# Optional: save a copy to Google Drive instead of downloading it.
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(OUTPUT_DIR, "/content/drive/MyDrive/superbot-lora", dirs_exist_ok=True)